In [1]:
import requests
import json
import os
from dotenv import load_dotenv

# 1. Carrega as variáveis escondidas no arquivo .env
load_dotenv()

# 2. Pega a chave de forma segura
minha_chave = os.getenv("API_KEY_FOOTBALL")

# 3. Definindo a URL e Parâmetros
url = "https://v3.football.api-sports.io/teams"
parametros = {
    "league": "71",
    "season": "2023"
}

# 4. O seu crachá de acesso usando a variável segura
headers = {
    "x-apisports-key": minha_chave
}

# 5. Fazendo a requisição
resposta = requests.get(url, headers=headers, params=parametros)
dados = resposta.json()

# 6. Validando o que chegou
if 'results' in dados and dados['results'] > 0:
    print(f"Sucesso! Total de times retornados pela API: {dados['results']}")
    primeiro_time = dados['response'][0]
    print(json.dumps(primeiro_time, indent=2))
else:
    print("Nenhum dado retornado ou erro na conexão. Verifique o retorno da API:")
    print(dados)

Sucesso! Total de times retornados pela API: 20
{
  "team": {
    "id": 118,
    "name": "Bahia",
    "code": "BAH",
    "country": "Brazil",
    "founded": 1931,
    "national": false,
    "logo": "https://media.api-sports.io/football/teams/118.png"
  },
  "venue": {
    "id": 216,
    "name": "Arena Fonte Nova",
    "address": "Rua Lions Club, Nazar\u00e9",
    "city": "Salvador, Bahia",
    "capacity": 56500,
    "surface": "grass",
    "image": "https://media.api-sports.io/football/venues/216.png"
  }
}


In [2]:
import pandas as pd

# 1. Isolando apenas a lista de times (que está dentro da chave 'response')
# 1. Isolando apenas a lista de times (que está dentro da chave 'response')
lista_times = dados['response']

# 2. Criando uma lista vazia que vai guardar nossas linhas tratadas
dados_estruturados = []

# 3. O "Trator": Passando time por time e extraindo só o que importa
for time in lista_times:
    linha = {
        "team_id": time['team']['id'],
        "nome": time['team']['name'],
        "cidade": time['venue']['city'],
        "ano_fundacao": time['team']['founded'],
        "estadio": time['venue']['name'] if 'venue' in time and time['venue'] else None
    }
    dados_estruturados.append(linha)

# 4. A Mágica do Pandas: Transformando a lista em um DataFrame (Tabela)
df_times = pd.DataFrame(dados_estruturados)

# 5. Exibindo as 5 primeiras linhas da nossa nova tabela limpa
print("Transformação concluída com sucesso! Veja a tabela:")
display(df_times.head()) # No Jupyter, o 'display' renderiza uma tabela visualmente mais bonita que o 'print'

Transformação concluída com sucesso! Veja a tabela:


,team_id,nome,cidade,ano_fundacao,estadio
0,118,Bahia,"Salvador, Bahia",1931,Arena Fonte Nova
1,119,Internacional,"Porto Alegre, Rio Grande do Sul",1909,Estádio José Pinheiro Borda
2,120,Botafogo,Rio de Janeiro,1904,Estádio Nilton Santos
3,121,Palmeiras,"São Paulo, São Paulo",1914,Allianz Parque
4,124,Fluminense,"Rio de Janeiro, Rio de Janeiro",1902,Estadio Jornalista Mário Filho (Maracanã)


In [3]:
import sqlite3

# 1. Estabelecendo a conexão (o Python cria o arquivo do banco automaticamente na sua pasta)
conexao = sqlite3.connect('banco_brasileirao.db')

# 2. A Carga: Enviando o DataFrame do Pandas direto para a tabela do banco
# O parâmetro if_exists='replace' garante que, se rodarmos amanhã, ele atualiza a tabela sem duplicar tudo
df_times.to_sql(name='dim_times', con=conexao, if_exists='replace', index=False)

# 3. Fechando a conexão por segurança
conexao.close()

print("Carga concluída com sucesso! Os dados estão blindados no banco local.")

Carga concluída com sucesso! Os dados estão blindados no banco local.
